# Functional Test 3: Ontology (`ontology-build` / `ontology-consume`)

## Summary

- The **ontology is built by the Python code of `ontology-build` and the Azure OpenAI chat model that it calls**. Foundry IQ only reads out the original passages. This notebook **tests the features** of `ontology-build` and `ontology-consume` from `foundry-iq-skills`, which you obtain from the official distribution source. The notebook does not bundle the runtime and does not download it automatically.
- The test has two parts.
  - **Part A (offline)**: Calls neither Azure nor a model. You import an OWL file in Turtle format and check hygiene, review, approval, export to JSON-LD, profile validation, and version diffs.
  - **Part B (live)**: Takes the output directory of a Topic Graph as input, and an Azure OpenAI chat model **proposes** an ontology. You check the estimate, approval, execution, and validation. Because the model is called, a small cost is incurred (the estimate appears in the result of `plan`, and you set the cap with `MAX_SPEND_USD`).
- In Part B, the input text (the articles on Thornwick Castle and Elowen Park) is used through the Topic Graph output that Functional Test 2 builds and the Foundry IQ index it came from. Part A prepares, inside the notebook, a small Turtle file that matches the content of the same two articles.
- When you run to the end with `Run All`, each test is recorded as PASS / FAIL and a list appears at the end. The Part A results also appear right after section 8. If you do not need Part B, stop there.

> Source: [Ontology build (foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-build/ontology-build.md) / [Connect, plan, propose](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-build/references/connect-plan-propose.md) / [Review, approve, export](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-build/references/review-approve-export.md) / [Check and estimate](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-consume/references/check-and-estimate.md)

## Test flow

```mermaid
flowchart LR
    subgraph A["Part A: Offline"]
        T["OWL in Turtle"] --> I["import<br/>hygiene"] --> R["review"] --> P["approve"] --> E["export<br/>JSON-LD"] --> C["consume check<br/>show / diff"]
    end
    subgraph B["Part B: Live"]
        O["Topic Graph<br/>output directory"] --> CN["connect"] --> PL["plan<br/>estimate"] --> PR["propose<br/>model call"] --> RV["review / approve<br/>export / check"]
    end
```

## Test list

| ID | Target | Pass condition |
| --- | --- | --- |
| T0 | Runtime | `setup ontology` returns `ready`. The CLI version is printed |
| A1 | `import` (valid Turtle) | 6 classes, 5 relationships, and 1 attribute are imported, and no items are dropped |
| A2 | `import` (invalid Turtle) | The dangling relationship is dropped by hygiene, and `subClassOf` is ignored |
| A3 | `review` | An approval card appears, and the version is not yet approved |
| A4 | `approve` | A wrong schema ID is rejected with `SCHEMA_ID_MISMATCH`, and the correct ID approves the version |
| A5 | `export` | JSON-LD is written, the SHA-256 matches, and an existing output path is not overwritten |
| A6 | `consume check` (valid) | 0 violations |
| A7 | `consume check` (invalid) | Three kinds of modification are each rejected by the expected rule |
| A8 | `consume show` / `current` | The lists of classes and relationships match, and the SHA-256 of the currently approved ontology matches |
| A9 | `consume diff` | The difference between version 1 and version 2 is only the added class and relationship |
| A10 | Datatype normalization | `xsd:integer` is normalized to `Double` (observed in 0.1.0) |
| B1 | `doctor` / `connect` | No check fails, and the Topic Graph output and the source index are validated |
| B2 | `inspect` / `plan` | Topics are found, and the estimate is at or below the cap |
| B3 | `propose` | A new version is created, and the cost is within the approved budget |
| B4 | Content of the proposal | There are classes and relationships, and every type has evidence |
| B5 | `review` → `approve` → `export` → `check` | The proposed version becomes JSON-LD that passes validation |

## Prerequisites

| Item | Details |
| --- | --- |
| Runtime | Obtain it from the official distribution source by following [the manual download and placement steps in the README](./README.md) (section 5.2). By default, you place it in the `foundry-iq-skills` folder at the same level as the notebook. If it is somewhere else, specify the location with the environment variable `FOUNDRY_IQ_PLUGIN_ROOT` |
| Python | Any version from 3.12 through 3.14. `setup` installs into a cache dedicated to the plugin and does not change the global environment |
| Part B | You are signed in with Azure CLI. An Azure OpenAI chat deployment. Read permission on the source index (Search Index Data Reader). `AZURE_OPENAI_ENDPOINT` and `AZURE_OPENAI_DEPLOYMENT` in `.env` |
| Part B input | The output of Functional Test 2 (`.functional-test/topicgraph-out`). If it does not exist, the notebook looks for earlier output under `.foundry-iq` |

## 1. Setup

### 1.1 Install the packages

The notebook itself uses only display and graph-drawing libraries. `ontology-build` itself is installed by the plugin into a dedicated environment (see 1.3 below).

In [ ]:
%pip install -q python-dotenv pandas networkx matplotlib azure-identity

### 1.2 Configuration

You use the working directory `.functional-test/ontology`. **The storage location of the approved ontology is kept inside this working directory with the environment variable `ONTOLOGY_HOME`**, so that the tests do not pollute the default location (`%LOCALAPPDATA%\ontology`).

`check()` records the result of each test in `results`. In normal checks, processing continues even on failure, and everything is shown together at the end. However, if the B3 proposal fails, the cell shows its response and stops.

In [ ]:
import base64
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)
matplotlib.rcParams["font.sans-serif"] = ["Segoe UI", "Arial", "DejaVu Sans"]

PLUGIN_ROOT = Path(os.environ.get("FOUNDRY_IQ_PLUGIN_ROOT", "foundry-iq-skills")).expanduser().resolve()
BOOTSTRAP = PLUGIN_ROOT / "graph" / "runtime" / "bootstrap.py"
if not BOOTSTRAP.is_file():
    raise FileNotFoundError(f"The Foundry IQ runtime is not installed: {BOOTSTRAP}. Follow the manual download and placement steps in the README (section 5.2): put foundry-iq-skills at the same level as the notebook, or set FOUNDRY_IQ_PLUGIN_ROOT to its location. The notebook does not download it automatically.")
WORK = Path(".functional-test/ontology").resolve()
WORK.mkdir(parents=True, exist_ok=True)
os.environ["ONTOLOGY_HOME"] = str(WORK / "home")

MAX_SPEND_USD = 1.0    # Cost cap (US dollars) for the Part B proposal
RUN_CLEANUP = False    # If True, deletes the working directory at the end

results = []


def check(case, ok, detail=""):
    results.append({"Test": case, "Result": "PASS" if ok else "FAIL", "Detail": str(detail)[:200]})
    print("PASS" if ok else "FAIL", "-", case, f"| {detail}" if detail else "")
    return ok


def cli(tool, *args, state=None):
    cmd = [sys.executable, "-I", str(BOOTSTRAP), "run", "ontology", tool, *map(str, args), "--json"]
    if state is not None:
        cmd += ["--state-dir", str(state)]
    proc = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
    try:
        data = json.loads(proc.stdout)
    except ValueError:
        data = {"raw": proc.stdout, "stderr": proc.stderr[-500:]}
    return proc.returncode, data


pd.DataFrame({"Value": [PLUGIN_ROOT, BOOTSTRAP.is_file(), WORK, os.environ["ONTOLOGY_HOME"]]},
             index=["Plugin", "bootstrap.py exists", "Working directory", "ONTOLOGY_HOME"])

### 1.3 T0: Prepare the runtime

`bootstrap.py setup ontology` verifies the hashes of the pinned wheels and the lock file, and installs them into an environment dedicated to the plugin (by default `%LOCALAPPDATA%\foundry-iq-skills\runtimes`). It calls neither Azure nor a model. The first run takes time because pip fetches the dependencies. `python -I` is the isolated mode, which disables the `PYTHON*` environment variables and the behavior of adding the script's location to `sys.path`.

> Source: [Install and checks (foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/graph-guide/references/install-and-checks.md) / [Python command line: -I](https://docs.python.org/3/using/cmdline.html#cmdoption-I)

In [ ]:
proc = subprocess.run([sys.executable, "-I", str(BOOTSTRAP), "setup", "ontology"], capture_output=True, text=True, encoding="utf-8", errors="replace")
setup = json.loads(proc.stdout.strip().splitlines()[-1])
check("T0-1 The ontology runtime is ready", proc.returncode == 0 and setup.get("status") == "ready", setup.get("python"))

version = subprocess.run([sys.executable, "-I", str(BOOTSTRAP), "run", "ontology", "ontology-build", "--version"],
                         capture_output=True, text=True, encoding="utf-8", errors="replace").stdout.strip()
check("T0-2 The ontology-build version is printed", version.startswith("ontology-build "), version)

## Part A: Offline tests

## 2. Prepare the input (OWL in Turtle)

OWL 2 is an ontology language that has classes, properties, individuals, and data values, and it is exchanged as RDF documents. Turtle is a form of that RDF that is easier for people to write. `ontology-build import` can read `.owl`, `.rdf`, `.xml`, `.ttl`, `.nt`, and `.json` files (it parses them with `rdflib` and similar libraries).

You prepare two inputs.

| File | Content | Purpose |
| --- | --- | --- |
| `hanapedia.ttl` | 6 classes, 5 object properties, and 1 datatype property | It imports cleanly |
| `bad.ttl` | Contains `subClassOf` and a `range` that points to a nonexistent class | Hygiene removes the items that do not fit the profile |

> Source: [OWL 2 Web Ontology Language Document Overview (W3C)](https://www.w3.org/TR/owl2-overview/) / [Review, approve, export (foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-build/references/review-approve-export.md)

In [ ]:
FIXTURES = WORK / "fixtures"
FIXTURES.mkdir(parents=True, exist_ok=True)

PREFIXES = """@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix : <https://example.com/hanapedia#> .

<https://example.com/hanapedia> a owl:Ontology ;
    rdfs:label "Hanapedia" .
"""

GOOD_TTL = PREFIXES + """
:HeritageSite a owl:Class ; rdfs:label "HeritageSite" ; rdfs:comment "A heritage site such as a castle, park or historic building. Examples: Elowen Park, Thornwick Castle" .
:Feature a owl:Class ; rdfs:label "Feature" ; rdfs:comment "A signature object, artwork or natural feature of a site. Examples: Twelve Stars Tapestry, Elowen Cherry" .
:Person a owl:Class ; rdfs:label "Person" ; rdfs:comment "A person. Examples: Osric Thorne, Queen Elizabeth I" .
:Facility a owl:Class ; rdfs:label "Facility" ; rdfs:comment "A building, garden or room within the grounds. Examples: Great Hall, Pavilion" .
:Place a owl:Class ; rdfs:label "Place" ; rdfs:comment "A place name or administrative area. Examples: Eastgate Ward, Aldermere" .
:Event a owl:Class ; rdfs:label "Event" ; rdfs:comment "A festival, ceremony or historical event. Examples: Blossom Festival, Great Fire of Aldermere" .

:locatedIn a owl:ObjectProperty ; rdfs:label "locatedIn" ; rdfs:domain :HeritageSite ; rdfs:range :Place ; rdfs:comment "The area where the site is located" .
:hasFeature a owl:ObjectProperty ; rdfs:label "hasFeature" ; rdfs:domain :HeritageSite ; rdfs:range :Feature ; rdfs:comment "A signature object or feature of the site" .
:foundedBy a owl:ObjectProperty ; rdfs:label "foundedBy" ; rdfs:domain :HeritageSite ; rdfs:range :Person ; rdfs:comment "The person who founded the site" .
:hasFacility a owl:ObjectProperty ; rdfs:label "hasFacility" ; rdfs:domain :HeritageSite ; rdfs:range :Facility ; rdfs:comment "A building or garden within the grounds" .
:survivedEvent a owl:ObjectProperty ; rdfs:label "survivedEvent" ; rdfs:domain :HeritageSite ; rdfs:range :Event ; rdfs:comment "An event, such as a fire, that the site escaped" .
:foundedYear a owl:DatatypeProperty ; rdfs:label "foundedYear" ; rdfs:domain :HeritageSite ; rdfs:range xsd:integer ; rdfs:comment "The calendar year in which the site was founded" .
"""

BAD_TTL = """@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix : <https://example.com/bad#> .

<https://example.com/bad> a owl:Ontology .

:Site a owl:Class ; rdfs:comment "Facility" .
:Castle a owl:Class ; rdfs:subClassOf :Site ; rdfs:comment "Castle" .
:Place a owl:Class ; rdfs:comment "Place" .
:locatedIn a owl:ObjectProperty ; rdfs:domain :Site ; rdfs:range :Missing ; rdfs:comment "Located in" .
"""

V2_TTL = GOOD_TTL + """
:Artifact a owl:Class ; rdfs:label "Artifact" ; rdfs:comment "A heritage asset held by a castle or park" .
:hasArtifact a owl:ObjectProperty ; rdfs:label "hasArtifact" ; rdfs:domain :HeritageSite ; rdfs:range :Artifact ; rdfs:comment "A heritage asset held by the site" .
"""

for name, text in {"hanapedia.ttl": GOOD_TTL, "bad.ttl": BAD_TTL, "hanapedia-v2.ttl": V2_TTL}.items():
    (FIXTURES / name).write_text(text, encoding="utf-8")
pd.DataFrame([{"File": p.name, "Bytes": p.stat().st_size} for p in sorted(FIXTURES.glob("*.ttl"))])

## 3. A1 / A2: `import` and hygiene

`import` reads a file, applies hygiene, which **drops or renames** the items that do not fit the profile, and saves the result as a new version (`version`). `hygiene.dropped` in the result is the number of dropped items. Even when `status` is `ok`, the content may have shrunk, so you check the counts.

The state directory (`--state-dir`) holds the records of versions, approvals, and exports. It is recreated for each test.

In [ ]:
STATE_A = WORK / "state-a"
STATE_BAD = WORK / "state-bad"
for folder in (STATE_A, STATE_BAD):
    shutil.rmtree(folder, ignore_errors=True)

rc, good = cli("ontology-build", "import", FIXTURES / "hanapedia.ttl", state=STATE_A)
check("A1 A valid Turtle file is imported", rc == 0 and (good["classes"], good["relationships"], good["attributes"]) == (6, 5, 1) and good["hygiene"]["dropped"] == 0,
      f"classes {good.get('classes')} / relationships {good.get('relationships')} / attributes {good.get('attributes')} / dropped {good.get('hygiene', {}).get('dropped')}")

rc, bad = cli("ontology-build", "import", FIXTURES / "bad.ttl", state=STATE_BAD)
check("A2 Invalid items are dropped by hygiene", rc == 0 and bad["relationships"] == 0 and bad["hygiene"]["dropped"] == 1 and bad["classes"] == 3,
      f"classes {bad.get('classes')} / relationships {bad.get('relationships')} / dropped {bad.get('hygiene', {}).get('dropped')}")

pd.DataFrame({"Valid input": {k: good.get(k) for k in ("version", "classes", "relationships", "attributes", "schema_id")},
              "Invalid input": {k: bad.get(k) for k in ("version", "classes", "relationships", "attributes", "schema_id")}})

## 4. A3 / A4: Review and approval

`review` shows the content of a version as plain counts (types, relationships, attributes) and returns an **approval card**. Approval is not a write to Azure. It is a review that says, "make this version the current ontology that later steps use."

You pass `approve` the **schema ID** (a hash of the content) that `review` showed. If the ID differs, the call is rejected with `SCHEMA_ID_MISMATCH`. This mechanism prevents the reviewed content and the approved content from being swapped.

In [ ]:
rc, review = cli("ontology-build", "review", "--version", 1, state=STATE_A)
check("A3 The review shows an approval card, and the version is not yet approved", rc == 0 and review["approved"] is False and review["approval"][0].startswith("APPROVAL"),
      review["approval"][1] if rc == 0 else review)
print("\n".join(review["approval"]))

SCHEMA_ID = review["schema_id"]
rc, wrong = cli("ontology-build", "approve", "--version", 1, "--schema-id", "sha256:deadbeef", state=STATE_A)
check("A4-1 A wrong schema ID is rejected", rc != 0 and wrong.get("error_code") == "SCHEMA_ID_MISMATCH", wrong.get("error_code"))

rc, approved = cli("ontology-build", "approve", "--version", 1, "--schema-id", SCHEMA_ID, state=STATE_A)
check("A4-2 The correct schema ID approves the version", rc == 0 and approved["status"] == "ok" and approved["already_approved"] is False, approved.get("approved_at"))

## 5. A5: Export to JSON-LD

`export` writes **only an approved version** to OWL JSON-LD (`.jsonld`). This is the format that you pass to `--ontology inline_jsonld --ontology-path` of Entity Graph. You check the following.

- The output is canonical JSON. The same content gives the same byte sequence.
- The SHA-256 recorded in the result matches the value computed from the file.
- An existing file is not overwritten unless you add `--force` (`OUTPUT_EXISTS`).

> Source: [JSON-LD 1.1 (W3C)](https://www.w3.org/TR/json-ld11/) / [Review, approve, export (foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-build/references/review-approve-export.md)

In [ ]:
OUT = WORK / "out"
shutil.rmtree(OUT, ignore_errors=True)
JSONLD = OUT / "hanapedia.jsonld"

rc, exported = cli("ontology-build", "export", "--version", 1, "--output", JSONLD, state=STATE_A)
actual_sha = hashlib.sha256(JSONLD.read_bytes()).hexdigest() if JSONLD.is_file() else None
check("A5-1 JSON-LD is written, and the SHA-256 matches", rc == 0 and exported["profile"] == "inline-jsonld-v1" and actual_sha == exported["jsonld_sha256"], exported.get("profile"))

rc, again = cli("ontology-build", "export", "--version", 1, "--output", JSONLD, state=STATE_A)
check("A5-2 The same output path is not overwritten", rc != 0 and again.get("error_code") == "OUTPUT_EXISTS", again.get("error_code"))

print(JSONLD.read_text(encoding="utf-8")[:600], "...")

## 6. A6 / A7: Profile validation (`ontology-consume check`)

`check` examines whether JSON-LD (or `.owl`, `.rdf`, `.xml`) conforms to the **ontology profile**, and returns the violations together with their rule names. You can use it as a pre-check before you connect to an Entity Graph knowledge source. The main rules are that the file is canonical, that each declaration is one of `owl:Class`, `owl:ObjectProperty`, and `owl:DatatypeProperty`, and that the `domain` and `range` of a property point to declared classes. Turtle is outside the scope of `check`. You `import` it first and then `export` it.

You apply the following three modifications to a valid file and confirm that each one is rejected.

| Modification | Expected rule |
| --- | --- |
| Change the `range` of a property to a nonexistent class | `property_range` |
| Change the type of a property to `owl:Restriction` | `declaration_type` |
| Pretty-print the file so that the byte sequence leaves the canonical form | `not_canonical` |

> Source: [Check and estimate (foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-consume/references/check-and-estimate.md)

In [ ]:
rc, ok = cli("ontology-consume", "check", JSONLD)
check("A6 A valid JSON-LD has 0 violations", rc == 0 and ok["status"] == "ok" and ok["violations"] == [], ok.get("violations"))

original = json.loads(JSONLD.read_text(encoding="utf-8"))


def dump(obj, pretty=False):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, **({"indent": 2} if pretty else {"separators": (",", ":")}))


dangling = json.loads(json.dumps(original))
next(n for n in dangling["@graph"] if n["@id"].endswith("#hasFeature"))["rdfs:range"] = {"@id": "urn:ontology:ontology#Ghost"}

wrong_type = json.loads(json.dumps(original))
next(n for n in wrong_type["@graph"] if n["@id"].endswith("#hasFeature"))["@type"] = "owl:Restriction"

NEGATIVES = {"property_range": dump(dangling), "declaration_type": dump(wrong_type), "not_canonical": dump(original, pretty=True)}
rows = []
for rule, text in NEGATIVES.items():
    path = OUT / f"negative-{rule}.jsonld"
    path.write_text(text, encoding="utf-8")
    rc, data = cli("ontology-consume", "check", path)
    violations = data.get("details", {}).get("violations", [])
    hit = any(v.startswith(rule) for v in violations)
    check(f"A7 The {rule} violation is detected", rc != 0 and data.get("error_code") == "PROFILE_VIOLATION" and hit, violations[:1])
    rows.append({"Modification": rule, "error_code": data.get("error_code"), "Violation": " / ".join(violations)})
pd.DataFrame(rows)

## 7. A8: Browse the content and the "currently approved" ontology

`ontology-consume show` returns the list of classes and properties, and `current` returns the path, SHA-256, and counts of the **currently approved ontology**. `current` recomputes the file hash on every call, so if the file has been rewritten, it stops with `CURRENT_INVALID`.

You draw the declarations from the import as a diagram, with classes as circles and object properties as arrows.

In [ ]:
rc, shown = cli("ontology-consume", "show", "--version", 1, state=STATE_A)
class_names = {c["name"] for c in shown["classes"]}
relations = {(p["domain"], p["name"], p["range"]) for p in shown["properties"] if p["kind"] == "object"}
EXPECTED_CLASSES = {"HeritageSite", "Feature", "Person", "Facility", "Place", "Event"}
EXPECTED_RELATIONS = {("HeritageSite", "locatedIn", "Place"), ("HeritageSite", "hasFeature", "Feature"), ("HeritageSite", "foundedBy", "Person"),
                      ("HeritageSite", "hasFacility", "Facility"), ("HeritageSite", "survivedEvent", "Event")}
check("A8-1 The classes and relationships match the input", rc == 0 and class_names == EXPECTED_CLASSES and relations == EXPECTED_RELATIONS, f"{len(class_names)} classes / {len(relations)} relationships")

rc, current = cli("ontology-consume", "current", state=STATE_A)
current_sha = hashlib.sha256(Path(current["path"]).read_bytes()).hexdigest()
check("A8-2 The SHA-256 of the currently approved ontology matches", rc == 0 and current["sha256"] == current_sha == actual_sha, current["sha256"][:16])


def draw_ontology(show, title):
    g = nx.DiGraph()
    g.add_nodes_from(c["name"] for c in show["classes"])
    g.add_edges_from((p["domain"], p["range"], {"predicate": p["name"]}) for p in show["properties"] if p["kind"] == "object")
    pos = nx.spring_layout(g, seed=7, k=1.4)
    fig, ax = plt.subplots(figsize=(9, 6))
    nx.draw_networkx_nodes(g, pos, ax=ax, node_size=1800, node_color="#9ecae1")
    nx.draw_networkx_edges(g, pos, ax=ax, arrows=True, arrowsize=14, edge_color="#888888", node_size=1800)
    nx.draw_networkx_labels(g, pos, ax=ax, font_size=8)
    nx.draw_networkx_edge_labels(g, pos, ax=ax, edge_labels=nx.get_edge_attributes(g, "predicate"), font_size=7)
    ax.set_title(title)
    ax.axis("off")
    plt.show()


draw_ontology(shown, "Imported ontology (version 1)")

## 8. A9 / A10: Version diff and datatype normalization

In the same state directory, you import a Turtle file that adds the class `Artifact` and the relationship `hasArtifact` as **version 2**, and you confirm that `diff` returns only the difference. Versions are appended and never overwritten.

You also confirm that `foundedYear`, which the Turtle file declares as `xsd:integer`, is treated as `Double` in the profile. This is presumably because the profile aligns the datatype range to `String`, `Double`, `Boolean`, and `DateTime` (the proposal prompt of `ontology-build` also lists the same four). This is an observation from 0.1.0.

> Source: [prompts.py (ontology_skills wheel)](https://github.com/microsoft/azure-skills/tree/main/.github/plugins/foundry-iq-skills/graph/runtime/wheels)

In [ ]:
rc, v2 = cli("ontology-build", "import", FIXTURES / "hanapedia-v2.ttl", state=STATE_A)
rc, diff = cli("ontology-consume", "diff", 1, 2, state=STATE_A)
added = diff["diff"]["added"]
check("A9 The version diff contains only the added items", rc == 0 and v2["version"] == 2 and added == {"classes": ["Artifact"], "properties": ["hasArtifact"]}
      and diff["diff"]["removed"] == {"classes": [], "properties": []}, added)

rc, shown2 = cli("ontology-consume", "show", "--version", 2, state=STATE_A)
founded_year = next(p for p in shown2["properties"] if p["name"] == "foundedYear")
check("A10 xsd:integer is normalized to Double", founded_year["range"] == "Double", f"kind={founded_year['kind']} range={founded_year['range']}")
draw_ontology(shown2, "Version 2 (Artifact added)")

### Part A results

These are the results of the tests so far. Part B (which uses Azure and a model) starts in the next chapter.

In [ ]:
part_a = pd.DataFrame(results)
print(part_a["Result"].value_counts().to_dict())
part_a

## Part B: Live tests (using Azure and a model)

## 9. Decide the input and the connection target

Part B checks that `ontology-build` **actually proposes an ontology**. The flow consists of the following five verbs.

| Verb | Azure | Model | Description |
| --- | --- | --- | --- |
| `connect` | Read only | Not called | Ties the state directory to the tenant, the Topic Graph output directory, and the model deployment. Reads the schema of the source index and pins it |
| `inspect` | Not called | Not called | Lists the topic levels and the default sampling |
| `plan` | Not called | Not called | **Estimates** the number of calls and the cost, and saves the plan. Returns an approval card |
| `propose` | Read only | **Called** | Runs the approved plan. Reads the selected passages and has the chat model make the proposal |
| `review` / `approve` / `export` | Not called | Not called | Finalizes the proposal with the same steps as in Part A |

The input `--topic-graph-output` is the directory that the Topic Graph native executable wrote to `outputDirectory`. **The location of the original source index is read from the record in this directory (`context.json`).** The text of the passages is read not at build time but from the **current** index (`--accept-current-source` is your consent to that).

> Source: [Connect, plan, propose (foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-build/references/connect-plan-propose.md)

In [ ]:
from azure.identity import DefaultAzureCredential

AOAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
CHAT_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]

token = DefaultAzureCredential().get_token("https://management.azure.com/.default").token
payload = token.split(".")[1]
TENANT_ID = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))["tid"]

# Model unit prices (US dollars per 1 million tokens). They are needed to enforce the cost cap. They are read from environment variables, or otherwise from earlier build records
PRICE_IN = os.environ.get("MODEL_INPUT_PRICE_PER_1M")
PRICE_OUT = os.environ.get("MODEL_OUTPUT_PRICE_PER_1M")
recorded = next(iter(sorted(Path(".foundry-iq").glob("*/entity/build-args.json"))), None)
if (PRICE_IN is None or PRICE_OUT is None) and recorded:
    PRICE_IN, PRICE_OUT = json.loads(recorded.read_text(encoding="utf-8"))["chat_model_usd_per_1m"]

candidates = [Path(".functional-test/topicgraph-out"), *sorted(Path(".foundry-iq").glob("*/topics/topicgraph-output/*"))]
TOPICGRAPH_OUTPUT = next((p.resolve() for p in candidates if (p / "topicGraph" / "run-manifest.json").is_file()), None)

assert TOPICGRAPH_OUTPUT, "The Topic Graph output was not found. Run Functional Test 2 first, or specify the path in TOPICGRAPH_OUTPUT"
assert PRICE_IN is not None and PRICE_OUT is not None, "The model unit prices are unknown. Set the environment variables MODEL_INPUT_PRICE_PER_1M and MODEL_OUTPUT_PRICE_PER_1M"

pd.DataFrame({"Value": [AOAI_ENDPOINT, CHAT_DEPLOYMENT, TOPICGRAPH_OUTPUT, PRICE_IN, PRICE_OUT]},
             index=["Azure OpenAI", "Chat deployment", "Topic Graph output", "Input price (USD per 1M tokens)", "Output price (USD per 1M tokens)"])

## 10. B1: `doctor` and `connect`

`doctor` is read only. It checks, in order, the installation, the sign-in, the Topic Graph output, the source index, and the model access. **It does not call the model.** It is normal for the output to say that chat access to the model is unverified until the first approved call.

Once `connect` succeeds, the connection is saved in the state directory. Later verbs use the same connection with `--state-dir` alone.

In [ ]:
STATE_B = WORK / "state-live"
shutil.rmtree(STATE_B, ignore_errors=True)

MODEL_FLAGS = ["--model-endpoint", AOAI_ENDPOINT, "--model-deployment", CHAT_DEPLOYMENT]
rc, doctor = cli("ontology-build", "doctor", "--tenant-id", TENANT_ID, *MODEL_FLAGS, state=STATE_B)
check("B1-1 No doctor check fails", rc == 0 and all(c["status"] != "fail" for c in doctor["checks"]), [c["name"] for c in doctor["checks"] if c["status"] == "fail"])
display(pd.DataFrame([{"Check": c["name"], "Status": c["status"], "Detail": c["detail"][:110]} for c in doctor["checks"]]))

In [ ]:
rc, connected = cli("ontology-build", "connect", "--tenant-id", TENANT_ID, "--topic-graph-output", TOPICGRAPH_OUTPUT, "--accept-current-source",
                    *MODEL_FLAGS, "--model-input-price-per-1m", PRICE_IN, "--model-output-price-per-1m", PRICE_OUT, state=STATE_B)
by_name = {c["name"]: c["status"] for c in connected["checks"]}
check("B1-2 connect succeeds, and the source index and the topics are validated",
      rc == 0 and connected.get("status") == "ok" and by_name.get("source_index") == "ok" and by_name.get("topicgraph_topics") == "ok",
      connected.get("source_index"))
pd.DataFrame([{"Check": c["name"], "Status": c["status"], "Detail": c["detail"][:110]} for c in connected["checks"]])

## 11. B2: `inspect` and `plan` (estimate)

`inspect` returns the number of topics at each topic level. **Level 0 is the broadest**, which is the same numbering as in Topic Graph. The default level of `plan` is the broadest level that has 20 or more topics. For a small output that has only one topic, level 0 was selected (an observation from earlier output).

Without calling the model, `plan` estimates the number of calls and the cost, and saves a plan ID. The `usd` value of the estimate is the **worst case, in which all tokens are priced at the higher of the input and output unit prices**, and `expected_usd` is the expected value, which prices input and output separately. If the worst case exceeds the cap, the run stops with `BUDGET_CAP_BELOW_ESTIMATE`.

In [ ]:
rc, inspected = cli("ontology-build", "inspect", state=STATE_B)
check("B2-1 Topic levels are found", rc == 0 and len(inspected["levels"]) >= 1 and inspected["levels"][0].get("topics", 0) >= 1,
      f"levels {len(inspected['levels'])} / default level {inspected.get('default_level')} / default passages {inspected.get('default_passages')}")
display(pd.DataFrame(inspected["levels"]))

rc, plan = cli("ontology-build", "plan", "--model-max-spend-usd", MAX_SPEND_USD, state=STATE_B)
check("B2-2 The estimate is at or below the cap", rc == 0 and plan["estimate"]["usd"] <= MAX_SPEND_USD,
      f"worst case ${plan['estimate']['usd']:.4f} / expected ${plan['estimate'].get('expected_usd', 0):.4f} / {plan.get('calls')} calls")
print("\n".join(plan["approval"]))

## 12. B3: `propose` (calls the model)

**This cell calls the Azure OpenAI chat model.** You pass `--approved-budget-usd` the worst-case estimate rounded up, and you approve the model calls with `--approve-model-calls`. These two flags correspond to Approve on the approval card. Calls that would exceed the cap are not executed.

Internally, two kinds of system prompts are used.

1. **PROPOSAL**: The model looks at the passages grouped by topic and proposes classes and relationships together with the IDs of their evidence.
2. **AUDIT**: The model merges the candidates obtained from the split proposals into one, checking them against the evidence (merging synonyms, removing types that are too broad, and so on).

Only when `RUN_INCOMPLETE` and an `SSLError` on the model connection are returned, the cell waits 2 seconds and then 4 seconds and resumes, at most 2 times, with the same plan ID, state directory, and budget. It does not disable certificate verification. For other errors, or when the retry limit is reached, the cell stops with a `RuntimeError` that contains the exit code and the original CLI response. Values that are returned only on success, such as `spent_usd`, are not referenced on failure.

After you fix the cause, re-run this B3 cell without changing the state directory or the plan ID. Calls that have completed are neither resent nor billed again. Re-running from B1 deletes the state directory, so do not run it when you resume.

> Source: [Connect, plan, propose (foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-build/references/connect-plan-propose.md) / [prompts.py (ontology_skills wheel)](https://github.com/microsoft/azure-skills/tree/main/.github/plugins/foundry-iq-skills/graph/runtime/wheels) / [Python built-in exceptions: RuntimeError](https://docs.python.org/3/library/exceptions.html#RuntimeError)

In [ ]:
import time

BUDGET = math.ceil(plan["estimate"]["usd"] * 100) / 100
for attempt in range(3):
    rc, proposed = cli("ontology-build", "propose", "--plan-id", plan["plan_id"], "--approved-budget-usd", BUDGET, "--approve-model-calls", state=STATE_B)
    tls_failed = proposed.get("error_code") == "RUN_INCOMPLETE" and proposed.get("error", "").endswith(": SSLError)")
    if rc == 0 or not tls_failed or attempt == 2:
        break
    delay = 2 ** (attempt + 1)
    print(f"TLS error on the model connection. Resuming with the same plan in {delay} seconds ({attempt + 1}/2).")
    time.sleep(delay)

if not check("B3-1 A new version is created", rc == 0 and proposed.get("status") == "ok" and proposed.get("version", 0) >= 1, f"version {proposed.get('version')}"):
    raise RuntimeError(f"ontology-build propose failed (exit code {rc}):\n{json.dumps(proposed, ensure_ascii=False, indent=2)}")
check("B3-2 The cost is within the approved budget", proposed["spent_usd"] <= proposed["approved_budget_usd"], f"${proposed['spent_usd']:.4f} / ${proposed['approved_budget_usd']}")
pd.DataFrame([{k: proposed.get(k) for k in ("version", "classes", "properties", "passages", "requested_passages", "spent_usd", "dropped", "warnings")}])

In [ ]:
rc, proposed = cli("ontology-build", "review", "--version", 1, state=STATE_B)

## 13. B4: Check the content of the proposal

Every proposed type must have **1 to 5 evidence IDs**. A type with no evidence is dropped by hygiene. `evidence` in `show` is the number of evidence items of each type. You list the classes and properties and draw them as a diagram.

In [ ]:
rc, live_shown = cli("ontology-consume", "show", "--version", proposed["version"], "--limit", 200, state=STATE_B)
classes_df = pd.DataFrame(live_shown["classes"], columns=["name", "evidence", "description"])
props_df = pd.DataFrame(live_shown["properties"], columns=["domain", "name", "range", "kind", "evidence"])

check("B4-1 There are at least 3 classes", len(classes_df) >= 3, f"{len(classes_df)} classes")
check("B4-2 There are at least 2 relationships (properties)", len(props_df) >= 2, f"{len(props_df)} properties")
check("B4-3 Every type has evidence", bool((classes_df["evidence"] >= 1).all() and (props_df["evidence"] >= 1).all()),
      f"without evidence: classes {int((classes_df['evidence'] < 1).sum())} / properties {int((props_df['evidence'] < 1).sum())}")
display(classes_df)
display(props_df)
draw_ontology(live_shown, "Proposed ontology")

### View the evidence passages

`ontology-consume evidence <type name>` returns the passages that served as the evidence for that type. This verb lets a person check where in the original text a proposal came from.

In [ ]:
rc, evidence = cli("ontology-consume", "evidence", classes_df["name"].iloc[0], "--limit", 3, state=STATE_B)
print(json.dumps(evidence, ensure_ascii=False, indent=2)[:2000])

## 14. B5: Finalize and pass the profile validation

With the same steps as in Part A, you run `review`, `approve`, and `export`, and you pass the exported JSON-LD through `check`. This test confirms that **the generated version is in a format that you can pass to Entity Graph as it is**.

In [ ]:
VERSION = proposed["version"]
LIVE_JSONLD = OUT / "proposed.jsonld"
rc, live_review = cli("ontology-build", "review", "--version", VERSION, state=STATE_B)
rc, live_approved = cli("ontology-build", "approve", "--version", VERSION, "--schema-id", live_review["schema_id"], state=STATE_B)
rc, live_export = cli("ontology-build", "export", "--version", VERSION, "--output", LIVE_JSONLD, state=STATE_B)
rc, live_check = cli("ontology-consume", "check", LIVE_JSONLD)

check("B5-1 The proposed version is approved and exported", live_approved["status"] == "ok" and live_export["profile"] == "inline-jsonld-v1", live_export.get("output"))
check("B5-2 The exported JSON-LD passes the profile validation", rc == 0 and live_check["status"] == "ok" and live_check["violations"] == [], live_check.get("violations"))
print("Next step: pass", LIVE_JSONLD, "to --ontology inline_jsonld --ontology-path of entity-graph-build")

## 15. Results list

This shows the counts of PASS and FAIL and the details of each FAIL. If you did not run Part B, the Part B rows are not recorded.

In [ ]:
summary = pd.DataFrame(results)
print(summary["Result"].value_counts().to_dict())
summary.style.apply(lambda row: ["background-color: #ffd6d6" if row["Result"] == "FAIL" else "" for _ in row], axis=1)

## 16. Cleanup (optional)

This deletes the working directory (the state directories, the exported files, and `ONTOLOGY_HOME`). The `ontology` runtime (the plugin's dedicated cache) is not deleted, because other procedures also use it. Nothing was created in Azure, so there is no cleanup on the Azure side.

In [ ]:
if RUN_CLEANUP:
    shutil.rmtree(WORK, ignore_errors=True)
    print("Deleted:", WORK)